# Multiple Linear Regression

## Data Loading & Exploration

In [44]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn import metrics


def section(title):
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)


# Read CSV
df = pd.read_csv("../Day 11/data/advertising.csv")

section("LAST 5 ROWS")
print(df.tail())

section("SHAPE")
print("Shape:", df.shape)

section("COLUMNS")
print("Columns:", df.columns.tolist())

section("DATA TYPES & INFO")
print(df.info())

section("MISSING VALUES")
print(df.isnull().sum())

section("STATISTICAL SUMMARY")
print(df.describe())

section("DUPLICATE ROWS")
print("Duplicate rows:", df.duplicated().sum())



LAST 5 ROWS
        TV  Radio  Newspaper  Sales
195   38.2    3.7       13.8    7.6
196   94.2    4.9        8.1   14.0
197  177.0    9.3        6.4   14.8
198  283.6   42.0       66.2   25.5
199  232.1    8.6        8.7   18.4

SHAPE
Shape: (200, 4)

COLUMNS
Columns: ['TV', 'Radio', 'Newspaper', 'Sales']

DATA TYPES & INFO
<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   TV         200 non-null    float64
 1   Radio      200 non-null    float64
 2   Newspaper  200 non-null    float64
 3   Sales      200 non-null    float64
dtypes: float64(4)
memory usage: 6.4 KB
None

MISSING VALUES
TV           0
Radio        0
Newspaper    0
Sales        0
dtype: int64

STATISTICAL SUMMARY
               TV       Radio   Newspaper       Sales
count  200.000000  200.000000  200.000000  200.000000
mean   147.042500   23.264000   30.554000   15.130500
std     85.854236   14.

## Feature and Target Selection

In [45]:
X = df[["TV", "Radio", "Newspaper"]]
y = df["Sales"]

print(X.shape)
print(y.shape)


(200, 3)
(200,)


## Train-Test Split

In [46]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training set shape:", X_train.shape, y_train.shape)
print("Testing set shape:", X_test.shape, y_test.shape)


Training set shape: (160, 3) (160,)
Testing set shape: (40, 3) (40,)


## Model Training

In [47]:
mr_model = LinearRegression()
mr_model.fit(X_train, y_train)

print("Model coefficients:", mr_model.coef_)  # (Weights for TV, Radio, and Newspaper)
print("Model intercept:", mr_model.intercept_)  # Intercept of the regression line

Model coefficients: [0.05450927 0.10094536 0.00433665]
Model intercept: 4.714126402214127


## Overfitting Check (Train vs Test R²)

In [48]:
# Overfitting check: Train and Test Scores
train_score = mr_model.score(X_train, y_train)
test_score = mr_model.score(X_test, y_test)

print(f"Training Score (R^2): {train_score:.4f}")
print(f"Testing Score (R^2): {test_score:.4f}")


Training Score (R^2): 0.9001
Testing Score (R^2): 0.9059


## RMSE Evaluation

In [49]:
# Train RMSE
train_predictions = mr_model.predict(X_train)
train_rmse = np.sqrt(metrics.mean_squared_error(y_train, train_predictions))

# Test RMSE
test_predictions = mr_model.predict(X_test)
test_rmse = np.sqrt(metrics.mean_squared_error(y_test, test_predictions))

print(f"Train RMSE: {train_rmse:.4f}")
print(f"Test RMSE: {test_rmse:.4f}")


Train RMSE: 1.6359
Test RMSE: 1.7052


## MAE Evaluation

In [50]:
print(f"Train MAE: {metrics.mean_absolute_error(y_train, train_predictions):.4f}")
print(f"Test MAE: {metrics.mean_absolute_error(y_test, test_predictions):.4f}")

Train MAE: 1.2344
Test MAE: 1.2748


## R² Score Summary

In [51]:
print(f"Train R2 Score: {mr_model.score(X_train, y_train):.4f}")
print(f"Test R2 Score: {mr_model.score(X_test, y_test):.4f}")

Train R2 Score: 0.9001
Test R2 Score: 0.9059


## Overfitting Analysis: RMSE, MAE, R² Comparison

In [52]:
# --- How to read RMSE/MAE/R2 to decide "overfitting or not" ---
# Overfitting = model memorizes training data instead of learning general patterns.
# Symptom: it does GREAT on train but MUCH WORSE on unseen test data.
# So the check is always the same shape: compare each metric's train value vs its test value.
#   - RMSE & MAE (error metrics, lower is better):
#       If Test RMSE/MAE >> Train RMSE/MAE -> errors balloon on unseen data -> overfitting.
#       If they're close -> model errs by roughly the same amount on both sets -> good generalization.
#   - R2 (goodness-of-fit, higher/closer to 1 is better):
#       If Train R2 >> Test R2 (big drop) -> model explains train variance well but fails on test -> overfitting.
#       If Train R2 ~= Test R2 (or test is even slightly higher) -> consistent performance -> no overfitting.
#       If both R2 scores are low -> that's UNDERFITTING instead (model too simple, not enough signal captured).

comparison = pd.DataFrame(
    {
        "Train": [train_rmse, metrics.mean_absolute_error(y_train, train_predictions), train_score],
        "Test": [test_rmse, metrics.mean_absolute_error(y_test, test_predictions), test_score],
    },
    index=["RMSE", "MAE", "R2"],
)
# The "Gap" column is the key diagnostic: a large positive gap on RMSE/MAE (test error much
# bigger than train error), or a large negative gap on R2 (test - train), signals overfitting.
comparison["Gap (Test - Train)"] = comparison["Test"] - comparison["Train"]

section("TRAIN VS TEST COMPARISON")
print(comparison)

# We use R2 as the deciding metric because it's on a fixed, interpretable scale (0 to 1),
# unlike RMSE/MAE which depend on the units of Sales. 0.05 is a rule-of-thumb threshold:
# a gap that small is normal randomness from the train/test split, not overfitting.
r2_gap = train_score - test_score  # positive -> train did better than test
if r2_gap > 0.05:
    # Train clearly outperforms test -> model learned train-specific noise -> overfitting.
    print(f"\nConclusion: R2 drops by {r2_gap:.4f} from train to test -> model is OVERFITTING.")
elif r2_gap < -0.05:
    # Test outperforming train by a lot is unusual -> often a sign of a small/unrepresentative
    # test split rather than a modeling problem -> worth investigating, not "overfitting".
    print(f"\nConclusion: Test R2 is higher than train R2 by {abs(r2_gap):.4f} -> unusual, check the split/data.")
else:
    # Gap is small in both directions, and RMSE/MAE tell the same story (see table above)
    # -> the model performs consistently on data it has and hasn't seen -> generalizes well.
    print(f"\nConclusion: RMSE, MAE, and R2 are close between train and test (R2 gap = {r2_gap:.4f}) -> model GENERALIZES WELL, no overfitting.")



TRAIN VS TEST COMPARISON
         Train      Test  Gap (Test - Train)
RMSE  1.635892  1.705215            0.069323
MAE   1.234416  1.274826            0.040410
R2    0.900142  0.905901            0.005760

Conclusion: RMSE, MAE, and R2 are close between train and test (R2 gap = -0.0058) -> model GENERALIZES WELL, no overfitting.


## Prediction on New Data

In [53]:
tv = 200
radio = 0
newspaper = 0

predictions_df = mr_model.predict(
    pd.DataFrame(
        {
            "TV": [tv],
            "Radio": [radio],
            "Newspaper": [newspaper],
        }
    )
)

print(
    f"Predicted Sales for TV={tv}, Radio={radio}, Newspaper={newspaper}: {predictions_df[0]:.4f}"
)

Predicted Sales for TV=200, Radio=0, Newspaper=0: 15.6160
